This notebook solves for the ground-state of Helium, by parameterizing the Slater determinant,
$$
\Psi(\boldsymbol{X}_1,\boldsymbol{X}_2,Z_{\rm{eff}}) = \psi_{1\rm{s}}(\boldsymbol{r}_1,Z_{\rm{eff}})\psi_{1\rm{s}}(\boldsymbol{r}_2,Z_{\rm{eff}}) \left(\frac{1}{\sqrt{2}}\uparrow_1\downarrow_2 -  \frac{1}{\sqrt{2}}\downarrow_1\uparrow_2 \right)
$$

where the bracketed spin term is a normalized Singlet state, and can effectively be ignored for the calculation of Hamiltonian matrix element. $Z_{\rm{eff}}$ is the effective charge felt by each of the electrons.

The Hamiltonian for a He-like system is,
$$
\mathcal{H} = -\frac{1}{2} \left(\nabla^2_1 + \nabla^2_2 \right) - \frac{Z}{r_1}- \frac{Z}{r_2} + \frac{1}{r_{12}},
$$
where $Z$ is the charge of the nucleus and not to be confused with $Z_{\rm{eff}}$. 

We will show symbollically in this notebook, that for the matrix element,
$$
\mathcal{E}(Z_{\rm{eff}}) = \langle \Psi \vert\mathcal{H}\vert \Psi \rangle,
$$
where $\Psi$ defined as above, a minimum value is obtained for 
$$
Z_{\rm{eff}} = Z - \frac{5}{16}.
$$

In [1]:
import sympy as sp

# Define symbolic variables
e, epsilon_0, a = sp.symbols("e epsilon_0 a", positive=True, real=True)
theta2, phi2 = sp.symbols("theta2 phi2", real=True)

psi1 = sp.symbols('psi1')

r1,r2         = sp.symbols('r1 r2',real=True,positive=True)
phi1,phi2     = sp.symbols('phi1 phi2',real=True)
theta1,theta2 = sp.symbols('theta1 theta2',real=True)


u = sp.symbols('u',real=True,positive=True)
Z_eff     = sp.symbols('Z_eff',real=True,positive=True)
Z         = sp.symbols('Z',real=True,positive=True)
E= sp.symbols('E')


psi = sp.Function('psi')



In [2]:
import sympy as sp
from sympy import symbols, Ynm
# 1. Define standard symbols and your function
r, theta, phi = sp.symbols('r theta phi', positive=True)

def attackSQRT(expr):
    #sympy struggles when sqrts are involved. 
    sqrt_map = {
        root: sp.sqrt(root.args[0].factor()) 
        for root in expr.find(sp.Pow) 
        if root.args[1] == sp.Rational(1, 2)
    }
    
    return expr.subs(sqrt_map)

def sphericalLaplacian(f,r,theta,phi):
    term_r = (1 / r**2) * sp.diff(r**2 * sp.diff(f, r), r)
    term_theta = (1 / (r**2 * sp.sin(theta))) * sp.diff(sp.sin(theta) * sp.diff(f, theta), theta)
    term_phi = (1 / (r**2 * sp.sin(theta)**2)) * sp.diff(f, phi, 2)
    laplacian = term_r + term_theta + term_phi
    sp.simplify(laplacian)
    return laplacian


def oneParticleIntegral(expr, r_var, theta_var, phi_var):
    # Assuming expr uses specific variables, or you can substitute them if needed:
    phiint   = sp.Integral(expr, (phi_var, 0, 2*sp.pi)).simplify()
    #print(phiint)
    thetaint = sp.Integral(phiint * sp.sin(theta_var), (theta_var, 0, sp.pi)).simplify()
    #print(thetaint)
    #The theta integral in the 2-electron integrals introduces sqrts 
    rint     = sp.Integral(attackSQRT(thetaint*r_var*r_var), (r_var, 0, sp.oo)).simplify()
    #print(rint)
    #print(rint)
    
    return rint

def twoParticleIntegral(expr):
    oneInt = oneParticleIntegral(expr,   r1, theta1, phi1)
    twoInt = oneParticleIntegral(oneInt, r2, theta2, phi2)
    return twoInt



In [3]:
Y = Ynm(1, 0, theta, phi)

In [4]:
Y

Ynm(1, 0, theta, phi)

In [5]:
#sphericalLaplacian(r*Y)

In [6]:
Ynm(0, 0, theta, phi).expand(func=True)**4

1/(16*pi**2)

In [7]:
def oneSorbital(r,theta,phi,Zeff):
    
    return 2 * sp.sqrt(Zeff**3) * sp.exp(-Zeff*r) *  Ynm(0, 0, theta, phi).expand(func=True)

In [8]:
oneSorbital(r2,theta2,phi2,Z_eff)

Z_eff**(3/2)*exp(-Z_eff*r2)/sqrt(pi)

In [9]:
He_trial_GS = oneSorbital(r1,theta1,phi1,Z_eff) * oneSorbital(r2,theta2,phi2,Z_eff) 

In [10]:
He_trial_GS

Z_eff**3*exp(-Z_eff*r1)*exp(-Z_eff*r2)/pi

In [11]:
from sympy import S

frac = S(1)/2
KEOperator = (sphericalLaplacian(He_trial_GS, r1, theta1, phi1) + sphericalLaplacian(He_trial_GS, r2, theta2, phi2)).simplify() * (-frac)

In [12]:
KEOperator

-Z_eff**4*(r1*(Z_eff*r2 - 2) + r2*(Z_eff*r1 - 2))*exp(-Z_eff*(r1 + r2))/(2*pi*r1*r2)

In [13]:
KEIntegral = twoParticleIntegral( KEOperator*He_trial_GS )
KEIntegral

Z_eff**2

In [14]:
twoParticleIntegral(1/r1 * He_trial_GS*He_trial_GS)

Z_eff

In [15]:
nuclearAttraction = -Z/r1 + -Z/r2
nuclearAttractionIntegral = twoParticleIntegral(nuclearAttraction*He_trial_GS*He_trial_GS)
nuclearAttractionIntegral

-2*Z*Z_eff

In [16]:
coulombRepulsion = 1/ (sp.sqrt(r1**2 + r2**2 - 2*r1*r2 * sp.cos(theta1)))
coulombRepulsionIntegral = twoParticleIntegral(coulombRepulsion*He_trial_GS*He_trial_GS)

In [17]:
coulombRepulsionIntegral

5*Z_eff/8

In [18]:
totalEnergy = KEIntegral + nuclearAttractionIntegral + coulombRepulsionIntegral

In [19]:
zeff_result = sp.solve(sp.Eq(totalEnergy.diff(Z_eff),0), Z_eff)[0]
zeff_result

Z - 5/16

In [20]:
screenParamEq = sp.Eq(Z_eff / Z, (zeff_result / Z).expand())
screenParamEq

Eq(Z_eff/Z, 1 - 5/(16*Z))

In [21]:
totalEnergy.subs(Z_eff,zeff_result).subs(Z,2).evalf()

-2.84765625000000

which is additionally the output energy of the autostructure run,

```
A.S. Be-like C structure - energies
 &SALGEB MXCONF=1 MXVORB=1  &END
 1 0 
  2 
 &SMINIM NZION=2 NLAM=1 RADOUT='YES' INCLUD=1 &END
 -1.0
!!-0.84375
```

where $-0.84375$ is the variationally determined value - corresponding to a nuclear charge of 2 * 0.84375 ~= 27/17 we found above.

In [22]:
screenParamEq.subs(Z,1)

Eq(Z_eff, 11/16)

In [23]:
#For the Hydrogen anion...
totalEnergy.subs(Z_eff,zeff_result).subs(Z,1).evalf()

-0.472656250000000

In [24]:
#Which is notably > -0.5 the ground energy of the H atom. The above ansatz is not enough to calculate the H anion.